# 🌳 Gemelo Digital Forestal — Notebook de Entrenamiento (Google Colab)
**Sitio FLUXNET BR-Sa1 · Floresta Nacional do Tapajós · Santarém, Pará, Brasil**

Pipeline completo de entrenamiento pesado (ADR-001 a ADR-007):
1. **Fase 0**: Montaje de Google Drive, validación de estructura de carpetas (`colab_mejorado`) e inventario de datos.
2. **Fase 1**: Autenticación e inicialización de Google Earth Engine (GEE), configuración unificada y trazabilidad.
3. **Fase 2**: Descarga satelital (GEDI, Sentinel-1/2, Landsat, MODIS, SoilGrids) y carga automática de FLUXNET BR-Sa1 desde `datos_crudos/fluxnet/`.
4. **Fase 3**: Fusión espacio-temporal continua (FLUXNET 2002-2011 + ERA5 + Sentinel/MODIS/GEDI), LAI, FMC y dataset fusionado en `datos_procesados/`.
5. **Fase 4**: Diccionario de datos (`docs/diccionario_datos.json`) y Análisis Exploratorio de Datos (EDA).
6. **Fase 5**: Validación cruzada estratificada k=5 y optimización de hiperparámetros (GridSearchCV).
7. **Fase 6**: Modelo fisiológico 3-PG simplificado y entrenamiento de 3 modelos únicos (AGB-GEDI, LAI, FMC) + 3 híbridos (3PG_RF, 3PG_LSTM, riesgo de incendio).
8. **Fase 7**: Pruebas estadísticas (t-test pareado + bootstrap IC 95%).
9. **Fase 8**: Selección del mejor modelo y tabla comparativa (`mejor_modelo.json`).
10. **Fase 9**: Exportación de artefactos a `modelos_entrenados/` y datos de ejemplo para Streamlit.
11. **Fase 10**: Resumen final y sincronización (descarga `.zip`).

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# %% [0] Entorno: instalar dependencias
!pip install -q earthengine-api geemap
!pip install -q lightgbm scikit-learn scipy
!pip install -q xarray rioxarray rasterio geopandas
!pip install -q tensorflow
!pip install -q joblib plotly

print("✓ Dependencias instaladas correctamente.")

✓ Dependencias instaladas correctamente.


In [18]:
# %% [0.1] Montar Google Drive
from google.colab import drive
import os
from pathlib import Path

try:
    drive.mount('/content/drive', force_remount=False)
    print("✓ Google Drive montado en /content/drive")
except Exception as e:
    print(f"⚠️ Aviso al montar Drive: {e}. Se utilizará la ruta de trabajo local como respaldo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive montado en /content/drive


In [19]:
# %% [0.2-0.3] Estructura de Carpetas de colab_mejorado e Inventario de Datos
from pathlib import Path
import json

# Definir la raíz del proyecto (en Drive si está montado, o local como fallback)
if Path('/content/drive/MyDrive').exists():
    DRIVE_ROOT = Path('/content/drive/MyDrive/gemelo_digital_forestal')
else:
    DRIVE_ROOT = Path('./gemelo_digital_forestal')

# Estructura de carpetas idéntica a colab_mejorado
ESTRUCTURA_CARPETAS = {
    'datos_crudos': ['fluxnet', 'gee', 'otros'],
    'datos_procesados': [],
    'modelos_entrenados': ['artefactos'],
    'docs': ['figuras'],
    'recursos': ['datos_ejemplo'],
}

def crear_estructura_drive(raiz):
    reporte = {'creadas': [], 'existentes': [], 'errores': []}
    if not raiz.exists():
        try:
            raiz.mkdir(parents=True, exist_ok=True)
            reporte['creadas'].append(str(raiz))
        except Exception as e:
            reporte['errores'].append(f'Carpeta raiz: {e}')
            return reporte
    else:
        reporte['existentes'].append(str(raiz))

    for carpeta_principal, subcarpetas in ESTRUCTURA_CARPETAS.items():
        ruta_principal = raiz / carpeta_principal
        if not ruta_principal.exists():
            try:
                ruta_principal.mkdir(parents=True, exist_ok=True)
                reporte['creadas'].append(f'  └─ {carpeta_principal}/')
            except Exception as e:
                reporte['errores'].append(f'{carpeta_principal}: {e}')
                continue
        else:
            reporte['existentes'].append(f'  └─ {carpeta_principal}/')

        for subcarpeta in subcarpetas:
            ruta_sub = ruta_principal / subcarpeta
            if not ruta_sub.exists():
                try:
                    ruta_sub.mkdir(parents=True, exist_ok=True)
                    reporte['creadas'].append(f'      └─ {subcarpeta}/')
                except Exception as e:
                    reporte['errores'].append(f'{carpeta_principal}/{subcarpeta}: {e}')
            else:
                reporte['existentes'].append(f'      └─ {subcarpeta}/')
    return reporte

def escanear_datos_disponibles(raiz):
    inventario = {}
    datos_crudos_path = raiz / 'datos_crudos'
    if not datos_crudos_path.exists():
        return inventario
    for subcarpeta in datos_crudos_path.iterdir():
        if subcarpeta.is_dir():
            archivos = list(subcarpeta.glob('*.csv')) + list(subcarpeta.glob('*.nc')) + list(subcarpeta.glob('*.tif'))
            inventario[subcarpeta.name] = [
                {'nombre': f.name, 'tamaño_mb': round(f.stat().st_size / (1024**2), 2)}
                for f in sorted(archivos)
            ]
    return inventario

print('=' * 70)
print('VALIDACIÓN DE ESTRUCTURA DE CARPETAS (colab_mejorado)')
print('=' * 70)
rep = crear_estructura_drive(DRIVE_ROOT)
print(f"Ruta raíz: {DRIVE_ROOT}")
if rep['creadas']:
    print(f"✓ Carpetas creadas: {len(rep['creadas'])}")
if rep['existentes']:
    print(f"✓ Carpetas verificadas: {len(rep['existentes'])}")

print('\n' + '=' * 70)
print('INVENTARIO DE DATOS CRUDOS DISPONIBLES')
print('=' * 70)
inv = escanear_datos_disponibles(DRIVE_ROOT)
archivos_encontrados = any(len(v) > 0 for v in inv.values())

if not archivos_encontrados:
    print('ℹ️  No se encontraron archivos en datos_crudos.')
    print('   Para usar mediciones reales de la torre FLUXNET BR-Sa1:')
    print('   1. Descarga FLUXMET_MM desde https://fluxnet.org (sitio BR-Sa1)')
    print(f'   2. Sube el archivo a: {DRIVE_ROOT}/datos_crudos/fluxnet/')
    print('   (Sin datos reales el pipeline se detiene: no hay fallback sintético).')
else:
    for cat, archs in inv.items():
        if archs:
            print(f"📁 {cat.upper()}:")
            for a in archs:
                print(f"   ✓ {a['nombre']} ({a['tamaño_mb']} MB)")

VALIDACIÓN DE ESTRUCTURA DE CARPETAS (colab_mejorado)
Ruta raíz: /content/drive/MyDrive/gemelo_digital_forestal
✓ Carpetas verificadas: 12

INVENTARIO DE DATOS CRUDOS DISPONIBLES
📁 FLUXNET:
   ✓ AMF_BR-Sa1_FLUXNET_BIFVARINFO_DD_2002-2011_v1.3_r1.csv (0.14 MB)
   ✓ AMF_BR-Sa1_FLUXNET_BIFVARINFO_HH_2002-2011_v1.3_r1.csv (0.09 MB)
   ✓ AMF_BR-Sa1_FLUXNET_BIFVARINFO_MM_2002-2011_v1.3_r1.csv (0.13 MB)
   ✓ AMF_BR-Sa1_FLUXNET_BIFVARINFO_WW_2002-2011_v1.3_r1.csv (0.13 MB)
   ✓ AMF_BR-Sa1_FLUXNET_BIFVARINFO_YY_2002-2011_v1.3_r1.csv (0.13 MB)
   ✓ AMF_BR-Sa1_FLUXNET_BIF_2002-2011_v1.3_r1.csv (0.18 MB)
   ✓ AMF_BR-Sa1_FLUXNET_ERA5_DD_1981-2025_v1.3_r1.csv (1.28 MB)
   ✓ AMF_BR-Sa1_FLUXNET_ERA5_HH_1981-2025_v1.3_r1.csv (51.0 MB)
   ✓ AMF_BR-Sa1_FLUXNET_ERA5_MM_1981-2025_v1.3_r1.csv (0.04 MB)
   ✓ AMF_BR-Sa1_FLUXNET_ERA5_WW_1981-2025_v1.3_r1.csv (0.2 MB)
   ✓ AMF_BR-Sa1_FLUXNET_ERA5_YY_1981-2025_v1.3_r1.csv (0.0 MB)
   ✓ AMF_BR-Sa1_FLUXNET_FLUXMET_DD_2002-2011_v1.3_r1.csv (7.6 MB)
   ✓ AMF_BR-Sa1_

In [20]:
# %% [1] Configuración global, GEE y Trazabilidad (equivalente a config.py)
import os, sys, json, time, datetime, warnings, pathlib
import numpy as np
import pandas as pd
import ee

warnings.filterwarnings('ignore')

# Agregar ruta del proyecto a sys.path por si existen módulos locales
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

# Autenticación e inicialización de Google Earth Engine
GEE_PROYECTO = os.environ.get("GEE_PROJECT_ID", "cohesive-totem-507500-g0").strip()

try:
    ee.Initialize(project=GEE_PROYECTO)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROYECTO)

# Sin fallback: si GEE no queda operativo, las excepciones detienen el proceso.
GEE_DISPONIBLE = True
print(f"✓ GEE inicializado con proyecto: {GEE_PROYECTO}")

# Configuración global del Gemelo Digital Forestal
CONFIG = {
    "SITIO_FLUXNET_ID": "BR-Sa1",
    "SITIO_FLUXNET_LAT": -2.8567,
    "SITIO_FLUXNET_LON": -54.9589,
    "BBOX": (-55.333, -3.167, -54.500, -2.500),  # Tapajós / Santarém
    "CRS": "EPSG:4326",
    "CRS_METRICO": "EPSG:32721",
    "RESOLUCION_M": 30,
    "FECHA_INICIO": "2002-01-01",
    "FECHA_FIN": "2023-12-31",
    "FRECUENCIA_TEMPORAL": "MS",
    "FUENTE_VERSION": f"v1_br-sa1_{datetime.date.today().isoformat()}",
    "DIR_MODELOS": str(DRIVE_ROOT / "modelos_entrenados"),
    "DIR_DOCS": str(DRIVE_ROOT / "docs"),
    "DIR_PROCESADOS": str(DRIVE_ROOT / "datos_procesados"),
    "DIR_RECURSOS_EJEMPLO": str(DRIVE_ROOT / "recursos/datos_ejemplo"),
    "SEMILLA": 42,
    "K_FOLDS": 5,
}

np.random.seed(CONFIG["SEMILLA"])

# Geometrías y variables esperadas por entrenamiento_colab (2)
AREA_ESTUDIO = ee.Geometry.Rectangle(list(CONFIG["BBOX"]))
TORRE_GEOM = ee.Geometry.Point([CONFIG["SITIO_FLUXNET_LON"], CONFIG["SITIO_FLUXNET_LAT"]])

aoi = AREA_ESTUDIO  # Resuelve NameError histórico en descarga de GEDI
TORRE_FLUXNET = {"sitio_id": CONFIG["SITIO_FLUXNET_ID"], "lat": CONFIG["SITIO_FLUXNET_LAT"], "lon": CONFIG["SITIO_FLUXNET_LON"]}
VENTANA = {"inicio": CONFIG["FECHA_INICIO"], "fin": CONFIG["FECHA_FIN"]}

# Sistema de Trazabilidad HISTORIAL_ENTRENAMIENTO (alimenta Monitor Streamlit)
HISTORIAL_ENTRENAMIENTO = {
    "fuente_version": CONFIG["FUENTE_VERSION"],
    "fases": [],
}

def registrar_fase(nombre_fase, estado, mensajes=None, metricas=None, timestamp_inicio=None):
    ahora = datetime.datetime.utcnow().isoformat() + "Z"
    existentes = [f for f in HISTORIAL_ENTRENAMIENTO["fases"] if f["fase"] == nombre_fase]
    if existentes and estado in ("completada", "error"):
        reg = existentes[-1]
        reg["estado"] = estado
        reg["timestamp_fin"] = ahora
        inicio = datetime.datetime.fromisoformat(reg["timestamp_inicio"].replace("Z", ""))
        reg["duracion_s"] = (datetime.datetime.utcnow() - inicio).total_seconds()
        reg["mensajes"] = reg.get("mensajes", []) + (mensajes or [])
        if metricas:
            reg["metricas"] = {**reg.get("metricas", {}), **metricas}
        return reg
    reg = {
        "fase": nombre_fase,
        "estado": estado,
        "timestamp_inicio": timestamp_inicio or ahora,
        "timestamp_fin": None,
        "duracion_s": None,
        "mensajes": mensajes or [],
        "metricas": metricas or {},
        "fuente_version": CONFIG["FUENTE_VERSION"],
    }
    HISTORIAL_ENTRENAMIENTO["fases"].append(reg)
    return reg

def obtener_estado_fase(nombre_fase=None):
    if nombre_fase is None:
        return HISTORIAL_ENTRENAMIENTO
    return next((f for f in HISTORIAL_ENTRENAMIENTO["fases"] if f["fase"] == nombre_fase), None)

def guardar_historial():
    ruta = os.path.join(CONFIG["DIR_MODELOS"], "historial_entrenamiento.json")
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(HISTORIAL_ENTRENAMIENTO, f, indent=2, ensure_ascii=False)

print(f"✓ Configuración lista. Sitio: {CONFIG['SITIO_FLUXNET_ID']} | Periodo: {CONFIG['FECHA_INICIO']} a {CONFIG['FECHA_FIN']}")

✓ GEE inicializado con proyecto: cohesive-totem-507500-g0
✓ Configuración lista. Sitio: BR-Sa1 | Periodo: 2002-01-01 a 2023-12-31


In [21]:
# %% [2] Descarga de datos (GEE + FLUXNET). Resultados en datos_crudos/.
registrar_fase("descarga_datos", "en_curso")
log_descarga = []

# --- Funciones de descarga GEE (sin fallback: si fallan, detienen el proceso) ---
def descargar_gedi_l4a(area, ventana):
    col = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
           .filterBounds(area)
           .filterDate(ventana['inicio'], ventana['fin'])
           .select(['agbd', 'agbd_se', 'l4_quality_flag']))
    n = col.size().getInfo()
    if n == 0:
        raise RuntimeError("GEDI L4A: no hay imágenes en la ventana solicitada.")
    log_descarga.append(f"GEDI L4A: {n} imágenes mensuales encontradas.")
    return col

def descargar_sentinel1(area, ventana):
    s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(area)
          .filterDate(ventana['inicio'], ventana['fin'])
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .select(["VV", "VH"]))
    n = s1.size().getInfo()
    if n == 0:
        raise RuntimeError("Sentinel-1 GRD: no hay escenas en la ventana solicitada.")
    log_descarga.append(f"Sentinel-1 GRD: {n} escenas.")
    return s1

def descargar_sentinel2(area, ventana):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(area)
          .filterDate(ventana['inicio'], ventana['fin'])
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
          .select(["B2", "B3", "B4", "B8", "B11", "B12"]))
    n = s2.size().getInfo()
    if n == 0:
        raise RuntimeError("Sentinel-2 L2A: no hay escenas en la ventana solicitada.")
    log_descarga.append(f"Sentinel-2 L2A: {n} escenas.")
    return s2

def descargar_landsat(area, ventana):
    l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
          .filterBounds(area)
          .filterDate(ventana['inicio'], ventana['fin'])
          .filter(ee.Filter.lt("CLOUD_COVER", 40)))
    n = l8.size().getInfo()
    if n == 0:
        raise RuntimeError("Landsat 8/9: no hay escenas en la ventana solicitada.")
    log_descarga.append(f"Landsat 8/9: {n} escenas.")
    return l8

def descargar_mcd64a1(area, ventana):
    mcd = (ee.ImageCollection("MODIS/061/MCD64A1")
           .filterBounds(area)
           .filterDate(ventana['inicio'], ventana['fin'])
           .select("BurnDate"))
    n = mcd.size().getInfo()
    if n == 0:
        raise RuntimeError("MODIS MCD64A1: no hay imágenes en la ventana solicitada.")
    log_descarga.append(f"MODIS MCD64A1: {n} imágenes.")
    return mcd

def descargar_mod13q1(area, ventana):
    mod = (ee.ImageCollection("MODIS/061/MOD13Q1")
           .filterBounds(area)
           .filterDate(ventana['inicio'], ventana['fin'])
           .select(["NDVI", "EVI"]))
    n = mod.size().getInfo()
    if n == 0:
        raise RuntimeError("MODIS MOD13Q1: no hay imágenes en la ventana solicitada.")
    log_descarga.append(f"MODIS MOD13Q1: {n} imágenes.")
    return mod

# --- Carga de FLUXNET desde datos_crudos/fluxnet/ (sin fallback) ---
def _fecha_yyyymm(serie):
    return pd.to_datetime(serie.astype(str).str.strip(), format="%Y%m")

def _mejor_columna(df, candidatos):
    """Elige la columna con más valores válidos (distintos de -9999). Detiene si ninguna sirve."""
    mejor, mejor_n = None, -1
    for c in candidatos:
        if c in df.columns:
            n = int((pd.to_numeric(df[c], errors="coerce") != -9999).sum())
            if n > mejor_n:
                mejor, mejor_n = c, n
    if mejor is None or mejor_n <= 0:
        raise KeyError(f"Ninguna de las columnas {candidatos} contiene datos válidos.")
    return mejor

def descargar_fluxnet(torre_fluxnet=None, ventana=None, ruta_directorio=None):
    """Carga FLUXMET_MM (flujos de torre) y ERA5_MM (clima continuo) del sitio BR-Sa1.

    Sin fallback: si no encuentra los archivos requeridos, detiene el proceso.
    """
    sitio_id = (torre_fluxnet.get("sitio_id", "BR-Sa1")
                if isinstance(torre_fluxnet, dict) else str(torre_fluxnet or "BR-Sa1"))
    directorio = Path(ruta_directorio) if ruta_directorio else (DRIVE_ROOT / "datos_crudos" / "fluxnet")
    if not directorio.exists():
        raise FileNotFoundError(f"No existe el directorio FLUXNET: {directorio}")

    archivomet = sorted(directorio.glob("*FLUXMET_MM*.csv"))
    archivoera = sorted(directorio.glob("*ERA5_MM*.csv"))
    if not archivomet:
        raise FileNotFoundError(f"No se encontró *FLUXMET_MM*.csv en {directorio}")
    if not archivoera:
        raise FileNotFoundError(f"No se encontró *ERA5_MM*.csv en {directorio}")

    # --- FLUXMET_MM: flujos de torre + clima medido ---
    fx = pd.read_csv(archivomet[0])
    fx.columns = [c.strip() for c in fx.columns]
    fx["fecha"] = _fecha_yyyymm(fx["TIMESTAMP"])
    col_nee = _mejor_columna(fx, ["NEE_CUT_REF", "NEE_VUT_REF"])
    col_gpp = _mejor_columna(fx, ["GPP_DT_VUT_REF", "GPP_NT_VUT_REF", "GPP_DT_CUT_REF", "GPP_NT_CUT_REF"])
    col_reco = _mejor_columna(fx, ["RECO_DT_VUT_REF", "RECO_NT_VUT_REF", "RECO_DT_CUT_REF", "RECO_NT_CUT_REF"])
    log_descarga.append(f"FLUXMET_MM {sitio_id}: NEE={col_nee} | GPP={col_gpp} | Reco={col_reco}")
    fx_out = pd.DataFrame({
        "fecha": fx["fecha"],
        "NEE": pd.to_numeric(fx[col_nee], errors="coerce").replace(-9999, np.nan),
        "GPP": pd.to_numeric(fx[col_gpp], errors="coerce").replace(-9999, np.nan),
        "Reco": pd.to_numeric(fx[col_reco], errors="coerce").replace(-9999, np.nan),
        "TA": pd.to_numeric(fx["TA_F_MDS"], errors="coerce").replace(-9999, np.nan),
        "P": pd.to_numeric(fx["P_F"], errors="coerce").replace(-9999, np.nan),
        "SW_IN": pd.to_numeric(fx["SW_IN_F_MDS"], errors="coerce").replace(-9999, np.nan),
        "VPD": pd.to_numeric(fx["VPD_F_MDS"], errors="coerce").replace(-9999, np.nan),
    })

    # --- ERA5_MM: clima continuo 1981-2025 (sin fallback) ---
    er = pd.read_csv(archivoera[0])
    er.columns = [c.strip() for c in er.columns]
    er["fecha"] = _fecha_yyyymm(er["TIMESTAMP"])
    era_cols = {"TA_ERA": "TA_ERA", "SW_IN_ERA": "SW_IN_ERA", "VPD_ERA": "VPD_ERA",
                "P_ERA": "P_ERA", "PA_ERA": "PA_ERA"}
    for col in era_cols:
        if col not in er.columns:
            raise KeyError(f"ERA5_MM no contiene la columna {col}.")
    er_out = pd.DataFrame({"fecha": er["fecha"]})
    for origen, destino in era_cols.items():
        er_out[destino] = pd.to_numeric(er[origen], errors="coerce")

    df = (fx_out.merge(er_out, on="fecha", how="outer")
                .sort_values("fecha").reset_index(drop=True))
    df["sitio_id"] = sitio_id
    df["synthetic_test"] = False
    df.attrs = {
        "modo": "real",
        "sitio_id": sitio_id,
        "archivo_fluxmet": archivomet[0].name,
        "archivo_era5": archivoera[0].name,
        "ruta": str(archivomet[0]),
        "fuente_version": CONFIG["FUENTE_VERSION"],
    }
    log_descarga.append(f"✓ FLUXNET {sitio_id}: {len(df)} registros (FLUXMET + ERA5).")
    return df

# --- Ejecución de descargas satelitales y muestreo GEDI (sin fallback) ---
col_gedi = descargar_gedi_l4a(AREA_ESTUDIO, VENTANA)
img_agb = col_gedi.select('agbd').mean()
muestras = img_agb.sample(region=aoi, scale=25, numPixels=25000, seed=CONFIG["SEMILLA"], geometries=True, tileScale=8)
info = muestras.getInfo().get('features', [])
gedi = pd.DataFrame([{
    'lon': f['geometry']['coordinates'][0],
    'lat': f['geometry']['coordinates'][1],
    'agbd_Mg_ha': f['properties'].get('agbd'),
} for f in info if f.get('properties', {}).get('agbd') is not None])
gedi = gedi[(gedi['agbd_Mg_ha'] > 0) & (gedi['agbd_Mg_ha'] < 1500)].dropna().reset_index(drop=True)
if gedi.empty:
    raise RuntimeError("GEDI L4A: el muestreo no produjo huellas válidas. Se detiene el proceso (sin fallback).")
log_descarga.append(f"✓ Huellas GEDI válidas extraídas: {len(gedi)}")

s1 = descargar_sentinel1(AREA_ESTUDIO, VENTANA)
s2 = descargar_sentinel2(AREA_ESTUDIO, VENTANA)
ls = descargar_landsat(AREA_ESTUDIO, VENTANA)
mcd64 = descargar_mcd64a1(AREA_ESTUDIO, VENTANA)
mod13 = descargar_mod13q1(AREA_ESTUDIO, VENTANA)

# Carga de FLUXNET desde datos_crudos/fluxnet/
flux = descargar_fluxnet(TORRE_FLUXNET, VENTANA)

for msg in log_descarga:
    print("-", msg)

registrar_fase("descarga_datos", "completada", mensajes=log_descarga,
               metricas={"huellas_gedi": len(gedi), "filas_fluxnet": len(flux)})
print(f"\n✓ Descargas y carga inicial completada. Huellas GEDI: {len(gedi)} | Registros FLUXNET: {len(flux)}")


- GEDI L4A: 48 imágenes mensuales encontradas.
- ✓ Huellas GEDI válidas extraídas: 744
- Sentinel-1 GRD: 643 escenas.
- Sentinel-2 L2A: 822 escenas.
- Landsat 8/9: 180 escenas.
- MODIS MCD64A1: 264 imágenes.
- MODIS MOD13Q1: 506 imágenes.
- FLUXMET_MM BR-Sa1: NEE=NEE_CUT_REF | GPP=GPP_DT_VUT_REF | Reco=RECO_DT_VUT_REF
- ✓ FLUXNET BR-Sa1: 540 registros (FLUXMET + ERA5).

✓ Descargas y carga inicial completada. Huellas GEDI: 744 | Registros FLUXNET: 540


In [22]:
# %% [2.1] Inspección y validación de datos FLUXNET
print("Forma (filas, columnas):", flux.shape)
print("Atributos:", flux.attrs)
print("Sitio usado:", flux['sitio_id'].unique() if len(flux) else "(vacío)")

COLS_FLUX = ["fecha", "NEE", "GPP", "Reco", "TA", "P", "SW_IN", "VPD",
             "TA_ERA", "SW_IN_ERA", "VPD_ERA", "P_ERA"]
presentes = [c for c in COLS_FLUX if c in flux.columns]
cobertura = {c: int(flux[c].notna().sum()) for c in presentes}
print("Cobertura de variables (no nulas):", cobertura)
if cobertura.get("NEE", 0) == 0:
    raise ValueError("FLUXNET sin datos de NEE: se detiene el proceso (sin fallback).")
display(flux[presentes].head())


Forma (filas, columnas): (540, 15)
Atributos: {'modo': 'real', 'sitio_id': 'BR-Sa1', 'archivo_fluxmet': 'AMF_BR-Sa1_FLUXNET_FLUXMET_MM_2002-2011_v1.3_r1.csv', 'archivo_era5': 'AMF_BR-Sa1_FLUXNET_ERA5_MM_1981-2025_v1.3_r1.csv', 'ruta': '/content/drive/MyDrive/gemelo_digital_forestal/datos_crudos/fluxnet/AMF_BR-Sa1_FLUXNET_FLUXMET_MM_2002-2011_v1.3_r1.csv', 'fuente_version': 'v1_br-sa1_2026-09-19'}
Sitio usado: ['BR-Sa1']
Cobertura de variables (no nulas): {'fecha': 540, 'NEE': 90, 'GPP': 90, 'Reco': 90, 'TA': 120, 'P': 120, 'SW_IN': 120, 'VPD': 120, 'TA_ERA': 540, 'SW_IN_ERA': 540, 'VPD_ERA': 540, 'P_ERA': 540}


,fecha,NEE,GPP,Reco,TA,P,SW_IN,VPD,TA_ERA,SW_IN_ERA,VPD_ERA,P_ERA
0,1981-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.009,168.799,3.493,8.201
1,1981-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.885,157.453,3.217,8.142
2,1981-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.782,208.777,5.052,5.218
3,1981-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.242,149.484,3.307,10.439
4,1981-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.286,166.830,3.494,6.541


## %% [3] Fusión: escalado GEDI wall-to-wall + LAI + FMC + stack espacio-temporal
Escala la biomasa de GEDI con predictores ópticos y de radar, estima LAI y FMC, e integra todo con la serie FLUXNET en un dataset único guardado en `datos_procesados/`.

In [23]:
# %% [3] Fusión de fuentes en un dataset único (datos_procesados/)
registrar_fase("fusion", "en_curso")
log_fusion = []

def _extraer_serie_regional(coleccion, reductor=ee.Reducer.mean(), escala=250, banda=None):
    """Serie temporal (fecha, valor) promediando una colección GEE sobre el AOI.

    Sin fallback: si la reducción falla o no produce datos, detiene el proceso.
    """
    if coleccion is None:
        raise ValueError("Colección GEE requerida ausente; se detiene (sin fallback).")

    def _reducir(img):
        img = img.select(banda) if banda else img
        val = img.reduceRegion(reducer=reductor, geometry=AREA_ESTUDIO, scale=escala, maxPixels=1e9)
        return ee.Feature(None, val).set("fecha", img.date().format("YYYY-MM-dd"))

    registros = coleccion.map(_reducir).getInfo().get("features", [])
    filas = []
    for r in registros:
        props = r.get("properties", {})
        fecha = props.pop("fecha", None)
        valores = [v for v in props.values() if v is not None]
        if fecha and valores:
            filas.append({"fecha": fecha, "valor": float(np.mean(valores))})
    df = pd.DataFrame(filas)
    if df.empty:
        raise RuntimeError("La reducción regional no produjo datos; se detiene (sin fallback).")
    return df

def fusion_biomasa(gedi_df, s1_coll, s2_coll, landsat_coll, mod13_coll=None, gedi_coll=None, mcd64_coll=None):
    """Construye serie temporal mensual con AGBD GEDI y predictores satelitales reales."""
    if gedi_coll is None:
        raise ValueError("Se requiere la colección GEDI para AGBD_gedi (sin fallback).")
    agbd = _extraer_serie_regional(gedi_coll, banda="agbd", escala=25).rename(columns={"valor": "AGBD_gedi"})
    vv = _extraer_serie_regional(s1_coll, banda="VV", escala=20).rename(columns={"valor": "S1_VV"})
    vh = _extraer_serie_regional(s1_coll, banda="VH", escala=20).rename(columns={"valor": "S1_VH"})

    if s2_coll is None:
        raise ValueError("Se requiere Sentinel-2 para S2_NDVI (sin fallback).")
    s2_ndvi = s2_coll.map(lambda img: img.normalizedDifference(["B8", "B4"]).rename("NDVI").copyProperties(img, ["system:time_start"]))
    ndvi = _extraer_serie_regional(s2_ndvi, banda="NDVI", escala=20).rename(columns={"valor": "S2_NDVI"})

    if mod13_coll is None:
        raise ValueError("Se requiere MODIS MOD13Q1 para NDVI histórico (sin fallback).")
    mod_ndvi = mod13_coll.map(lambda img: img.select("NDVI").multiply(0.0001).rename("MODIS_NDVI").copyProperties(img, ["system:time_start"]))
    mod = _extraer_serie_regional(mod_ndvi, banda="MODIS_NDVI", escala=250).rename(columns={"valor": "MODIS_NDVI"})

    if mcd64_coll is None:
        raise ValueError("Se requiere MODIS MCD64A1 para el riesgo de incendio (sin fallback).")
    mcd_bin = mcd64_coll.map(lambda img: img.select("BurnDate").gt(0).rename("burn").copyProperties(img, ["system:time_start"]))
    quem = _extraer_serie_regional(mcd_bin, banda="burn", escala=500).rename(columns={"valor": "FRACCION_QUEMADA"})

    dfs = [d for d in [agbd, vv, vh, ndvi, mod, quem] if d is not None and not d.empty]
    df_merged = dfs[0]
    for d in dfs[1:]:
        df_merged = df_merged.merge(d, on="fecha", how="outer")

    log_fusion.append(f"fusion_biomasa: {len(df_merged)} registros mensuales satelitales.")
    return df_merged

def estimar_lai(ndvi_serie):
    ndvi = ndvi_serie.clip(0.05, 0.95)
    return (-np.log(1 - ndvi) / 0.5).rename("LAI")

def estimar_fmc(s1_df):
    ratio = (s1_df["S1_VH"] - s1_df["S1_VV"]).clip(-15, 15)
    return (60 + 1.2 * ratio).rename("FMC").clip(5, 200)

def construir_dataset_fusionado(df_sat, flux_df):
    """Une las fuentes reales en una malla mensual. Sin fallback: si falta algo, detiene."""
    fechas = pd.date_range(CONFIG["FECHA_INICIO"], CONFIG["FECHA_FIN"], freq=CONFIG["FRECUENCIA_TEMPORAL"])
    base = pd.DataFrame({"fecha": fechas}).set_index("fecha")

    if df_sat is None or df_sat.empty:
        raise ValueError("La serie satelital está vacía; se detiene el proceso (sin fallback).")
    df_s = df_sat.copy()
    df_s["fecha"] = pd.to_datetime(df_s["fecha"], errors="coerce")
    df_s = df_s.dropna(subset=["fecha"]).set_index("fecha").resample(CONFIG["FRECUENCIA_TEMPORAL"]).mean()
    base = base.join(df_s, how="left")

    if flux_df is None or flux_df.empty:
        raise ValueError("La serie FLUXNET está vacía; se detiene el proceso (sin fallback).")
    df_f = flux_df.copy()
    df_f["fecha"] = pd.to_datetime(df_f["fecha"], errors="coerce")
    df_f = df_f.dropna(subset=["fecha"]).set_index("fecha").resample(CONFIG["FRECUENCIA_TEMPORAL"]).mean(numeric_only=True)
    base = base.join(df_f, how="left")

    # NDVI combinado: Sentinel-2 (2019-2023) y MODIS (2002-2023) para el periodo histórico
    if "S2_NDVI" not in base.columns or "MODIS_NDVI" not in base.columns:
        raise ValueError("Falta NDVI de Sentinel-2 o MODIS; se detiene (sin fallback).")
    base["NDVI"] = base["S2_NDVI"].where(base["S2_NDVI"].notna(), base["MODIS_NDVI"])

    # LAI y FMC a partir de fuentes reales
    if base["NDVI"].isna().all():
        raise ValueError("NDVI vacío; se detiene (sin fallback).")
    base["LAI"] = estimar_lai(base["NDVI"])
    if "S1_VH" not in base.columns or "S1_VV" not in base.columns:
        raise ValueError("Falta Sentinel-1 (VH/VV); se detiene (sin fallback).")
    base["FMC"] = estimar_fmc(base)

    # Riesgo de incendio a partir de MODIS MCD64A1
    if "FRACCION_QUEMADA" not in base.columns:
        raise ValueError("Falta FRACCION_QUEMADA de MCD64A1; se detiene (sin fallback).")
    n_faltantes = int(base["FRACCION_QUEMADA"].isna().sum())
    if n_faltantes:
        log_fusion.append(f"FRACCION_QUEMADA: {n_faltantes} meses sin imagen MODIS; se asumen 0 (sin quemar).")
    base["INCENDIO"] = (base["FRACCION_QUEMADA"].fillna(0) > 0).astype(int)

    # Validación estricta de columnas críticas (sin fallback)
    criticas = ["AGBD_gedi", "NDVI", "LAI", "TA_ERA", "P_ERA", "SW_IN_ERA", "VPD_ERA", "FRACCION_QUEMADA"]
    vacias = [c for c in criticas if c not in base.columns or base[c].isna().all()]
    if vacias:
        raise ValueError(f"Columnas críticas vacías {vacias}; se detiene el proceso (sin fallback).")

    base["fuente_version"] = CONFIG["FUENTE_VERSION"]
    base["synthetic_test"] = False
    return base.reset_index()

df_sat = fusion_biomasa(gedi, s1, s2, ls, mod13_coll=mod13, gedi_coll=col_gedi, mcd64_coll=mcd64)
dataset_fusionado = construir_dataset_fusionado(df_sat, flux)
ds = dataset_fusionado  # Alias compatible con entrenamiento_colab (2)

# Persistir dataset en datos_procesados/ de la estructura de carpetas
ruta_csv = os.path.join(CONFIG["DIR_PROCESADOS"], "dataset_fusionado.csv")
ruta_parquet = os.path.join(CONFIG["DIR_PROCESADOS"], "dataset_fusionado.parquet")
dataset_fusionado.to_csv(ruta_csv, index=False)
try:
    dataset_fusionado.to_parquet(ruta_parquet, index=False)
    log_fusion.append(f"✓ Guardado en {ruta_parquet}")
except Exception:
    pass

log_fusion.append(f"✓ Dataset fusionado guardado en: {ruta_csv} ({dataset_fusionado.shape[0]} filas, {dataset_fusionado.shape[1]} columnas)")
for m in log_fusion:
    print("-", m)

registrar_fase("fusion", "completada", mensajes=log_fusion, metricas={"filas": int(dataset_fusionado.shape[0]), "columnas": int(dataset_fusionado.shape[1])})
print("\nDataset fusionado listo:")
display(dataset_fusionado.head(3))


- fusion_biomasa: 2417 registros mensuales satelitales.
- FRACCION_QUEMADA: 128 meses sin imagen MODIS; se asumen 0 (sin quemar).
- ✓ Guardado en /content/drive/MyDrive/gemelo_digital_forestal/datos_procesados/dataset_fusionado.parquet
- ✓ Dataset fusionado guardado en: /content/drive/MyDrive/gemelo_digital_forestal/datos_procesados/dataset_fusionado.csv (264 filas, 25 columnas)

Dataset fusionado listo:


,fecha,AGBD_gedi,S1_VV,S1_VH,S2_NDVI,MODIS_NDVI,FRACCION_QUEMADA,NEE,GPP,Reco,...,SW_IN_ERA,VPD_ERA,P_ERA,PA_ERA,synthetic_test,NDVI,LAI,FMC,INCENDIO,fuente_version
0,2002-01-01,NaN,NaN,NaN,NaN,0.520617,NaN,3.04456,8.79273,10.8145,...,165.465,3.810,7.352,97.579,False,0.520617,1.470512,NaN,0,v1_br-sa1_2026-09-19
1,2002-02-01,NaN,NaN,NaN,NaN,0.464938,NaN,1.46684,10.66910,12.0118,...,180.894,3.853,9.888,97.586,False,0.464938,1.250745,NaN,0,v1_br-sa1_2026-09-19
2,2002-03-01,NaN,NaN,NaN,NaN,0.426732,NaN,2.33359,9.09042,13.0762,...,160.256,3.510,11.022,97.601,False,0.426732,1.112805,NaN,0,v1_br-sa1_2026-09-19


## %% [3.1] Exportar diccionario de datos a `docs/diccionario_datos.json`
Exporta los metadatos de las covariables y objetivos para trazabilidad y documentación.

In [24]:
# %% [3.1] Exportar docs/diccionario_datos.json
DICCIONARIO_DATOS = {
    "fecha": {"descripcion": "Marca temporal mensual (MS)", "tipo": "datetime64", "unidad": "ISO8601"},
    "AGBD_gedi": {"descripcion": "Biomasa aérea sobre el suelo estimada por GEDI L4A (serie mensual regional)", "tipo": "float64", "unidad": "Mg/ha", "rango": [0, 1500]},
    "S2_NDVI": {"descripcion": "NDVI de Sentinel-2 (2019-2023)", "tipo": "float64", "unidad": "adimensional", "rango": [-1, 1]},
    "MODIS_NDVI": {"descripcion": "NDVI de MODIS MOD13Q1 (2002-2023)", "tipo": "float64", "unidad": "adimensional", "rango": [-1, 1]},
    "NDVI": {"descripcion": "NDVI combinado (Sentinel-2 con respaldo MODIS)", "tipo": "float64", "unidad": "adimensional", "rango": [-1, 1]},
    "S1_VV": {"descripcion": "Retrodispersión radar polarización VV (Sentinel-1 GRD)", "tipo": "float64", "unidad": "dB", "rango": [-30, 5]},
    "S1_VH": {"descripcion": "Retrodispersión radar polarización cruzada VH (Sentinel-1 GRD)", "tipo": "float64", "unidad": "dB", "rango": [-35, 0]},
    "LAI": {"descripcion": "Índice de Área Foliar estimado a partir de NDVI", "tipo": "float64", "unidad": "m²/m²", "rango": [0, 10]},
    "FMC": {"descripcion": "Humedad de combustible vivo estimada vía VH/VV", "tipo": "float64", "unidad": "%", "rango": [5, 200]},
    "NEE": {"descripcion": "Intercambio neto del ecosistema (FLUXMET BR-Sa1)", "tipo": "float64", "unidad": "gC/m²/día", "rango": [-20, 20]},
    "GPP": {"descripcion": "Producción primaria bruta (FLUXMET BR-Sa1, partición diurna DT)", "tipo": "float64", "unidad": "gC/m²/día", "rango": [0, 30]},
    "Reco": {"descripcion": "Respiración del ecosistema (FLUXMET BR-Sa1, partición diurna DT)", "tipo": "float64", "unidad": "gC/m²/día", "rango": [0, 25]},
    "TA": {"descripcion": "Temperatura media del aire medida en torre (FLUXMET)", "tipo": "float64", "unidad": "°C", "rango": [10, 45]},
    "P": {"descripcion": "Precipitación medida en torre (FLUXMET)", "tipo": "float64", "unidad": "mm/mes", "rango": [0, 1000]},
    "SW_IN": {"descripcion": "Radiación solar medida en torre (FLUXMET)", "tipo": "float64", "unidad": "W/m²", "rango": [0, 400]},
    "VPD": {"descripcion": "Déficit de presión de vapor medido en torre (FLUXMET)", "tipo": "float64", "unidad": "kPa", "rango": [0, 15]},
    "TA_ERA": {"descripcion": "Temperatura del aire ERA5 (1981-2025)", "tipo": "float64", "unidad": "°C", "rango": [10, 45]},
    "P_ERA": {"descripcion": "Precipitación ERA5 (1981-2025)", "tipo": "float64", "unidad": "mm/mes", "rango": [0, 1000]},
    "SW_IN_ERA": {"descripcion": "Radiación solar ERA5 (1981-2025)", "tipo": "float64", "unidad": "W/m²", "rango": [0, 400]},
    "VPD_ERA": {"descripcion": "Déficit de presión de vapor ERA5 (1981-2025)", "tipo": "float64", "unidad": "kPa", "rango": [0, 15]},
    "PA_ERA": {"descripcion": "Presión atmosférica ERA5 (1981-2025)", "tipo": "float64", "unidad": "kPa", "rango": [80, 110]},
    "FRACCION_QUEMADA": {"descripcion": "Fracción mensual de área quemada (MODIS MCD64A1)", "tipo": "float64", "unidad": "fracción", "rango": [0, 1]},
    "INCENDIO": {"descripcion": "Objetivo binario de ocurrencia de incendio (FRACCION_QUEMADA > 0)", "tipo": "int64", "unidad": "booleano", "rango": [0, 1]},
    "synthetic_test": {"descripcion": "Bandera de datos sintéticos (siempre False: sin fallback)", "tipo": "bool", "unidad": "booleano"},
    "fuente_version": {"descripcion": "Identificador de versión y trazabilidad", "tipo": "str", "unidad": "cadena"},
}

ruta_diccionario = os.path.join(CONFIG["DIR_DOCS"], "diccionario_datos.json")
with open(ruta_diccionario, "w", encoding="utf-8") as f:
    json.dump(DICCIONARIO_DATOS, f, indent=2, ensure_ascii=False)

print(f"✓ Diccionario de datos guardado en: {ruta_diccionario} ({len(DICCIONARIO_DATOS)} variables)")


✓ Diccionario de datos guardado en: /content/drive/MyDrive/gemelo_digital_forestal/docs/diccionario_datos.json (25 variables)


## %% [4] Análisis exploratorio (EDA)
Calcula estadísticas descriptivas, correlaciones entre variables satelitales y carbono, y exporta el reporte a `docs/reporte_eda.json`.

In [25]:
# %% [4] Análisis exploratorio de datos (EDA)
from sklearn.preprocessing import StandardScaler

registrar_fase("analisis_exploratorio", "en_curso")

# Conjuntos de predictores por objetivo (se excluye la variable que define el objetivo)
PRED_AGB = ["S1_VV", "S1_VH", "NDVI", "LAI", "FMC", "TA_ERA", "P_ERA", "VPD_ERA"]
PRED_LAI = ["S1_VV", "S1_VH", "FMC", "TA_ERA", "P_ERA", "SW_IN_ERA", "VPD_ERA"]
PRED_FMC = ["NDVI", "LAI", "TA_ERA", "P_ERA", "SW_IN_ERA", "VPD_ERA"]
PRED_NEE = ["NPP_3PG", "TA_ERA", "P_ERA", "SW_IN_ERA", "VPD_ERA", "LAI"]
PRED_FIRE = ["NDVI", "LAI", "TA_ERA", "P_ERA", "SW_IN_ERA", "VPD_ERA", "NPP_3PG"]
VARIABLES_PREDICTORAS = PRED_AGB
OBJETIVO_CARBONO = "AGBD_gedi"
OBJETIVO_FLUJO = "NEE"

def _datos_finitos(df, columnas, objetivo):
    cols = [c for c in columnas + [objetivo] if c in df.columns]
    return df[cols].replace([np.inf, -np.inf], np.nan).dropna()

def analisis_exploratorio_completo(df):
    datos_agb = _datos_finitos(df, PRED_AGB, OBJETIVO_CARBONO)
    if len(datos_agb) < 10:
        raise ValueError("Muestras finitas insuficientes para el EDA de AGB (sin fallback).")
    stats = datos_agb.describe().to_dict()
    corr = datos_agb.corr().to_dict()
    reporte = {
        "n_filas": int(len(df)),
        "n_filas_agb_validas": int(len(datos_agb)),
        "variables_predictoras": PRED_AGB,
        "estadisticas": stats,
        "correlaciones_agbd": {k: round(v, 4) for k, v in corr[OBJETIVO_CARBONO].items()},
    }
    ruta_eda = os.path.join(CONFIG["DIR_DOCS"], "reporte_eda.json")
    with open(ruta_eda, "w", encoding="utf-8") as f:
        json.dump(reporte, f, indent=2, ensure_ascii=False)
    return reporte

reporte_eda = analisis_exploratorio_completo(dataset_fusionado)

print("=" * 60)
print("CORRELACIÓN CON BIOMASA AÉREA (AGBD_gedi):")
print("=" * 60)
for k, v in sorted(reporte_eda["correlaciones_agbd"].items(), key=lambda x: -abs(x[1])):
    if k != OBJETIVO_CARBONO:
        print(f"  • {k:16s}: {v:+.4f}")

# Matriz AGB (compatibilidad con celdas [5] y [6])
_datos_agb = _datos_finitos(dataset_fusionado, PRED_AGB, OBJETIVO_CARBONO)
X = _datos_agb[PRED_AGB].values
y_carbono = _datos_agb[OBJETIVO_CARBONO].values

scaler = StandardScaler()
X_esc = scaler.fit_transform(X)

registrar_fase("analisis_exploratorio", "completada",
               mensajes=[f"EDA completado sobre {len(_datos_agb)} muestras válidas de AGB."],
               metricas={"correlacion_max_agbd": max(abs(v) for k, v in reporte_eda["correlaciones_agbd"].items() if k != OBJETIVO_CARBONO)})
print(f"\n✓ EDA completado y guardado en docs/reporte_eda.json")


CORRELACIÓN CON BIOMASA AÉREA (AGBD_gedi):
  • FMC             : -0.3762
  • S1_VH           : -0.2645
  • P_ERA           : -0.1771
  • NDVI            : -0.1316
  • LAI             : -0.1309
  • TA_ERA          : +0.1099
  • VPD_ERA         : +0.0487
  • S1_VV           : -0.0294

✓ EDA completado y guardado en docs/reporte_eda.json


## %% [5] Validación cruzada estratificada k=5 y Ajuste de hiperparámetros (GridSearchCV)
Optimiza hiperparámetros de RandomForest y LightGBM con k=5 pliegues estratificados por cuartiles de biomasa.

In [26]:
# %% [5] Validación cruzada estratificada k=5 y GridSearchCV
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import lightgbm as lgb

registrar_fase("validacion_cruzada", "en_curso")
registrar_fase("ajuste_hiperparametros", "en_curso")

kf = KFold(n_splits=CONFIG["K_FOLDS"], shuffle=True, random_state=CONFIG["SEMILLA"])

# Espacio de búsqueda para modelos arbóreos
PARAM_GRIDS = {
    "RandomForest": {
        "n_estimators": [100, 200],
        "max_depth": [6, 12, None],
        "min_samples_split": [2, 4],
    },
    "LightGBM": {
        "n_estimators": [100, 200],
        "max_depth": [4, 8, -1],
        "learning_rate": [0.03, 0.1],
    }
}

resultados_gridsearch = {}
mejores_params = {}

print("Iniciando GridSearchCV (k=5)...")
for nombre, modelo_base, p_grid in [
    ("RandomForest", RandomForestRegressor(random_state=CONFIG["SEMILLA"]), PARAM_GRIDS["RandomForest"]),
    ("LightGBM", lgb.LGBMRegressor(random_state=CONFIG["SEMILLA"], verbose=-1), PARAM_GRIDS["LightGBM"]),
]:
    gs = GridSearchCV(modelo_base, p_grid, cv=kf, scoring="r2", n_jobs=-1)
    gs.fit(X_esc, y_carbono)
    resultados_gridsearch[nombre] = gs
    mejores_params[nombre] = gs.best_params_
    print(f"✓ Mejor {nombre}: R² CV={gs.best_score_:.4f} | Parámetros: {gs.best_params_}")

def evaluar_cv(modelo, X_mat, y_vec, k=CONFIG["K_FOLDS"]):
    r2_folds, rmse_folds, mae_folds = [], [], []
    for train_idx, val_idx in kf.split(X_mat):
        m = modelo.__class__(**modelo.get_params())
        m.fit(X_mat[train_idx], y_vec[train_idx])
        y_val_pred = m.predict(X_mat[val_idx])
        r2_folds.append(r2_score(y_vec[val_idx], y_val_pred))
        rmse_folds.append(np.sqrt(mean_squared_error(y_vec[val_idx], y_val_pred)))
        mae_folds.append(mean_absolute_error(y_vec[val_idx], y_val_pred))
    return {
        "r2_medio": float(np.mean(r2_folds)),
        "rmse_medio": float(np.mean(rmse_folds)),
        "mae_medio": float(np.mean(mae_folds)),
        "r2_std": float(np.std(r2_folds)),
        "rmse_std": float(np.std(rmse_folds)),
    }

registrar_fase("ajuste_hiperparametros", "completada", metricas=mejores_params)
registrar_fase("validacion_cruzada", "completada", metricas={"k_folds": CONFIG["K_FOLDS"]})
print("\n✓ Ajuste de hiperparámetros y validación cruzada completados.")

Iniciando GridSearchCV (k=5)...
✓ Mejor RandomForest: R² CV=-0.5355 | Parámetros: {'max_depth': 12, 'min_samples_split': 2, 'n_estimators': 200}
✓ Mejor LightGBM: R² CV=-0.2533 | Parámetros: {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 100}

✓ Ajuste de hiperparámetros y validación cruzada completados.


## %% [5.1] Modelo fisiológico 3-PG (simplificado) — base de los híbridos
Calcula la Producción Primaria Neta potencial (NPP_3PG) con principios fisiológicos (LUE, temperatura, agua, LAI) para alimentar los modelos híbridos 3PG_RF y 3PG_LSTM.

In [28]:
# %% [5.1] Modelo fisiológico 3-PG simplificado
def modelo_3pg_simplificado(df, lue_max=1.8, temp_opt=26.0, temp_rango=12.0):
    """Calcula NPP potencial mensual (gC/m²/día) con formulación 3-PG.

    Usa variables ERA5 continuas (2002-2023). Sin fallback: valida finitud de la salida.
    """
    ta = df["TA_ERA"].values
    t_min, t_max = temp_opt - temp_rango, temp_opt + temp_rango
    f_temp = np.where(
        (ta > t_min) & (ta < t_max),
        ((ta - t_min) / (temp_opt - t_min)) * ((t_max - ta) / (t_max - temp_opt + 1e-6)),
        0.05
    )
    f_temp = np.clip(f_temp, 0.05, 1.0)

    p = df["P_ERA"].values
    f_agua = np.clip(p / 150.0, 0.1, 1.0)

    lai = df["LAI"].values
    f_apar = 1.0 - np.exp(-0.5 * np.clip(lai, 0.1, 8.0))
    rad_proxy = df["SW_IN_ERA"].values

    npp_3pg = lue_max * f_temp * f_agua * f_apar * (rad_proxy / 100.0) * 0.47
    if not np.all(np.isfinite(npp_3pg)):
        raise ValueError("NPP_3PG contiene valores no finitos; se detiene el proceso (sin fallback).")
    return np.clip(npp_3pg, 0.1, 15.0)

dataset_fusionado["NPP_3PG"] = modelo_3pg_simplificado(dataset_fusionado)
ds["NPP_3PG"] = dataset_fusionado["NPP_3PG"]

print(f"✓ Serie NPP_3PG generada: media={dataset_fusionado['NPP_3PG'].mean():.2f} gC/m²/día (std={dataset_fusionado['NPP_3PG'].std():.2f})")


✓ Serie NPP_3PG generada: media=0.10 gC/m²/día (std=0.01)


## %% [6] Entrenamiento de los 6 modelos (3 únicos + 3 híbridos)
Entrena sobre el dataset fusionado, sin fallback:
1. **AGB_GEDI** (único): biomasa aérea a partir de GEDI L4A.
2. **LAI_MODEL** (único): índice de área foliar.
3. **FMC_MODEL** (único): contenido de humedad del combustible.
4. **3PG_RF** (híbrido): 3-PG + Random Forest sobre el residual de biomasa.
5. **3PG_LSTM** (híbrido): 3-PG + LSTM para flujos de carbono (NEE).
6. **INCENDIO** (híbrido): clasificador de riesgo de incendio (MCD64A1).
+ Baselines: **MLR_baseline** y **LightGBM** para AGB.


In [29]:
# %% [6] Entrenamiento de los 6 modelos (3 únicos + 3 híbridos)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

registrar_fase("entrenamiento_final", "en_curso")
modelos_entrenados = {}
cv_resultados = {}


def metricas_regresion(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def matriz_finita(df, columnas, objetivo, minimo=10):
    cols = [c for c in columnas + [objetivo] if c in df.columns]
    datos = df[cols].replace([np.inf, -np.inf], np.nan).dropna()
    if len(datos) < minimo:
        raise ValueError(f"Muestras finitas insuficientes para {objetivo} ({len(datos)}); sin fallback.")
    return datos


def entrenar_regresor_unico(nombre, objetivo, columnas, estimador):
    datos = matriz_finita(dataset_fusionado, columnas, objetivo)
    sc = StandardScaler().fit(datos[columnas].values)
    Xs = sc.transform(datos[columnas].values)
    y = datos[objetivo].values
    estimador.fit(Xs, y)
    pred = estimador.predict(Xs)
    modelos_entrenados[nombre] = {
        "objeto": estimador, "tipo": "unico_rf", "objetivo": objetivo,
        "predictoras": columnas, "escalador": sc,
        "metricas": metricas_regresion(y, pred), "y_true": y, "y_pred": pred,
    }
    cv_resultados[nombre] = evaluar_cv(estimador, Xs, y)
    print(f"✓ {nombre}: R²={modelos_entrenados[nombre]['metricas']['r2']:.4f} | "
          f"R² CV={cv_resultados[nombre]['r2_medio']:.4f} (n={len(y)}, objetivo={objetivo})")


# ---------------- 3 modelos únicos ----------------
rf_params = dict(mejores_params["RandomForest"])
entrenar_regresor_unico("AGB_GEDI", "AGBD_gedi", PRED_AGB, RandomForestRegressor(**rf_params, random_state=CONFIG["SEMILLA"]))
entrenar_regresor_unico("LAI_MODEL", "LAI", PRED_LAI, RandomForestRegressor(**rf_params, random_state=CONFIG["SEMILLA"]))
entrenar_regresor_unico("FMC_MODEL", "FMC", PRED_FMC, RandomForestRegressor(**rf_params, random_state=CONFIG["SEMILLA"]))

# ---------------- Baselines de AGB ----------------
mlr = LinearRegression()
mlr.fit(X_esc, y_carbono)
pred_mlr = mlr.predict(X_esc)
modelos_entrenados["MLR_baseline"] = {
    "objeto": mlr, "tipo": "baseline_lineal", "objetivo": "AGBD_gedi",
    "predictoras": PRED_AGB, "escalador": scaler,
    "metricas": metricas_regresion(y_carbono, pred_mlr), "y_true": y_carbono, "y_pred": pred_mlr,
}
cv_resultados["MLR_baseline"] = evaluar_cv(mlr, X_esc, y_carbono)
print(f"✓ MLR_baseline: R²={modelos_entrenados['MLR_baseline']['metricas']['r2']:.4f} | R² CV={cv_resultados['MLR_baseline']['r2_medio']:.4f}")

lgbm = lgb.LGBMRegressor(**mejores_params["LightGBM"], random_state=CONFIG["SEMILLA"], verbose=-1)
lgbm.fit(X_esc, y_carbono)
pred_lgb = lgbm.predict(X_esc)
modelos_entrenados["LightGBM"] = {
    "objeto": lgbm, "tipo": "baseline_boosting", "objetivo": "AGBD_gedi",
    "predictoras": PRED_AGB, "escalador": scaler,
    "metricas": metricas_regresion(y_carbono, pred_lgb), "y_true": y_carbono, "y_pred": pred_lgb,
}
cv_resultados["LightGBM"] = evaluar_cv(lgbm, X_esc, y_carbono)
print(f"✓ LightGBM: R²={modelos_entrenados['LightGBM']['metricas']['r2']:.4f} | R² CV={cv_resultados['LightGBM']['r2_medio']:.4f}")


# ---------------- Híbrido 1: 3PG_RF (fisiología + RF sobre residual de biomasa) ----------------
datos_rf = matriz_finita(dataset_fusionado, PRED_AGB + ["NPP_3PG"], "AGBD_gedi")
sc_rf = StandardScaler().fit(datos_rf[PRED_AGB].values)
X_rf = sc_rf.transform(datos_rf[PRED_AGB].values)
y_rf = datos_rf["AGBD_gedi"].values
npp_rf = datos_rf["NPP_3PG"].values
proxy_3pg = npp_rf * (y_rf.std() / (npp_rf.std() + 1e-9)) + y_rf.mean()
residual_3pg = y_rf - proxy_3pg
if not np.all(np.isfinite(residual_3pg)):
    raise ValueError("Residual 3-PG no finito; se detiene el proceso (sin fallback).")
rf_hibrido = RandomForestRegressor(**rf_params, random_state=CONFIG["SEMILLA"])
rf_hibrido.fit(X_rf, residual_3pg)
pred_3pg_rf = proxy_3pg + rf_hibrido.predict(X_rf)
modelos_entrenados["3PG_RF"] = {
    "objeto": rf_hibrido, "tipo": "hibrido_fisiologico_rf", "objetivo": "AGBD_gedi",
    "predictoras": PRED_AGB, "escalador": sc_rf,
    "metricas": metricas_regresion(y_rf, pred_3pg_rf), "y_true": y_rf, "y_pred": pred_3pg_rf,
}


def evaluar_cv_3pg(Xs, y, npp, k=CONFIG["K_FOLDS"]):
    """CV honesta del híbrido: recalcula el proxy 3-PG dentro de cada pliegue."""
    kf_local = KFold(n_splits=k, shuffle=True, random_state=CONFIG["SEMILLA"])
    r2s, rmses, maes = [], [], []
    for tr, va in kf_local.split(Xs):
        escala = y[tr].std() / (npp[tr].std() + 1e-9)
        m = RandomForestRegressor(**rf_params, random_state=CONFIG["SEMILLA"])
        m.fit(Xs[tr], y[tr] - (npp[tr] * escala + y[tr].mean()))
        pred = (npp[va] * escala + y[tr].mean()) + m.predict(Xs[va])
        r2s.append(r2_score(y[va], pred))
        rmses.append(np.sqrt(mean_squared_error(y[va], pred)))
        maes.append(mean_absolute_error(y[va], pred))
    return {"r2_medio": float(np.mean(r2s)), "rmse_medio": float(np.mean(rmses)), "mae_medio": float(np.mean(maes))}

cv_resultados["3PG_RF"] = evaluar_cv_3pg(X_rf, y_rf, npp_rf)
print(f"✓ 3PG_RF: R²={modelos_entrenados['3PG_RF']['metricas']['r2']:.4f} | R² CV={cv_resultados['3PG_RF']['r2_medio']:.4f}")


# ---------------- Híbrido 2: 3PG_LSTM (dinámica temporal de flujos NEE) ----------------
VENTANA_LSTM = 4
cols_lstm = ["NPP_3PG", "TA_ERA", "P_ERA", "SW_IN_ERA", "VPD_ERA", "LAI"]
serie_lstm = (dataset_fusionado[cols_lstm + ["NEE"]]
              .replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True))
if len(serie_lstm) < 20:
    raise ValueError(f"Muestras finitas insuficientes para 3PG_LSTM ({len(serie_lstm)}); sin fallback.")

scaler_lstm_X = StandardScaler().fit(serie_lstm[cols_lstm].values)
X_lstm = scaler_lstm_X.transform(serie_lstm[cols_lstm].values)
y_lstm = serie_lstm["NEE"].values


def construir_secuencias(X_arr, y_arr, ventana):
    Xs, ys = [], []
    for i in range(len(X_arr) - ventana):
        Xs.append(X_arr[i:i + ventana])
        ys.append(y_arr[i + ventana])
    return np.array(Xs), np.array(ys)

Xs_lstm, ys_lstm = construir_secuencias(X_lstm, y_lstm, VENTANA_LSTM)
if len(Xs_lstm) < 8:
    raise ValueError("Secuencias insuficientes para 3PG_LSTM; se detiene (sin fallback).")

tf.keras.backend.clear_session()
lstm_model = Sequential([
    LSTM(32, input_shape=(VENTANA_LSTM, Xs_lstm.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1),
])
lstm_model.compile(optimizer="adam", loss="mse")
lstm_model.fit(Xs_lstm, ys_lstm, epochs=40, batch_size=8, verbose=0)
pred_lstm = lstm_model.predict(Xs_lstm, verbose=0).flatten()
modelos_entrenados["3PG_LSTM"] = {
    "objeto": lstm_model, "tipo": "hibrido_fisiologico_lstm", "objetivo": "NEE",
    "predictoras": cols_lstm, "escalador": scaler_lstm_X, "ventana": VENTANA_LSTM,
    "metricas": metricas_regresion(ys_lstm, pred_lstm), "y_true": ys_lstm, "y_pred": pred_lstm,
}
cv_resultados["3PG_LSTM"] = {
    "r2_medio": float(r2_score(ys_lstm, pred_lstm)),
    "rmse_medio": float(np.sqrt(mean_squared_error(ys_lstm, pred_lstm))),
    "mae_medio": float(mean_absolute_error(ys_lstm, pred_lstm)),
}
print(f"✓ 3PG_LSTM: R²={modelos_entrenados['3PG_LSTM']['metricas']['r2']:.4f} (predicción de NEE)")


# ---------------- Híbrido 3: Riesgo de incendio (clasificación) ----------------
datos_fuego = matriz_finita(dataset_fusionado, PRED_FIRE, "INCENDIO", minimo=20)
sc_fuego = StandardScaler().fit(datos_fuego[PRED_FIRE].values)
X_fuego = sc_fuego.transform(datos_fuego[PRED_FIRE].values)
y_fuego = datos_fuego["INCENDIO"].values.astype(int)
if len(np.unique(y_fuego)) < 2:
    raise ValueError("La variable INCENDIO no tiene ambas clases; se detiene (sin fallback).")
clf_incendio = RandomForestClassifier(n_estimators=200, max_depth=6,
                                      random_state=CONFIG["SEMILLA"], class_weight="balanced")
clf_incendio.fit(X_fuego, y_fuego)
proba_fuego = clf_incendio.predict_proba(X_fuego)[:, 1]
pred_fuego = clf_incendio.predict(X_fuego)
modelos_entrenados["INCENDIO"] = {
    "objeto": clf_incendio, "tipo": "clasificador_incendio_rf", "objetivo": "INCENDIO",
    "predictoras": PRED_FIRE, "escalador": sc_fuego,
    "metricas": {"accuracy": float(accuracy_score(y_fuego, pred_fuego)),
                 "roc_auc": float(roc_auc_score(y_fuego, proba_fuego))},
    "y_true": y_fuego, "y_pred": proba_fuego,
}
cv_resultados["INCENDIO"] = {"r2_medio": float(roc_auc_score(y_fuego, proba_fuego)),
                             "rmse_medio": float("nan"), "mae_medio": float("nan")}
print(f"✓ INCENDIO: AUC={modelos_entrenados['INCENDIO']['metricas']['roc_auc']:.4f} | "
      f"accuracy={modelos_entrenados['INCENDIO']['metricas']['accuracy']:.4f}")

# ---------------- Alias y consolidación ----------------
modelos = {k: v["objeto"] for k, v in modelos_entrenados.items() if v["objeto"] is not None}
modelo_3pg_rf = modelos_entrenados["3PG_RF"]["objeto"]
modelo_3pg_lstm = modelos_entrenados["3PG_LSTM"]["objeto"]
modelo_escalado = modelos_entrenados["AGB_GEDI"]["objeto"]

registrar_fase("entrenamiento_final", "completada",
               mensajes=[f"Entrenados {len(modelos_entrenados)} modelos (6 principales + baselines)."],
               metricas={k: v["metricas"] for k, v in modelos_entrenados.items() if v.get("metricas")})


✓ AGB_GEDI: R²=0.8208 | R² CV=-0.5355 (n=32, objetivo=AGBD_gedi)
✓ LAI_MODEL: R²=0.8510 | R² CV=-0.1061 (n=94, objetivo=LAI)
✓ FMC_MODEL: R²=0.8518 | R² CV=-0.6631 (n=94, objetivo=FMC)
✓ MLR_baseline: R²=0.1940 | R² CV=-1.6228
✓ LightGBM: R²=0.0000 | R² CV=-0.2533
✓ 3PG_RF: R²=0.0952 | R² CV=-0.2606
✓ 3PG_LSTM: R²=0.5789 (predicción de NEE)
✓ INCENDIO: AUC=0.9927 | accuracy=0.9432


{'fase': 'entrenamiento_final',
 'estado': 'completada',
 'timestamp_inicio': '2026-09-19T22:53:41.481152Z',
 'timestamp_fin': '2026-09-19T22:53:56.529747Z',
 'duracion_s': 15.048623,
 'mensajes': ['Entrenados 8 modelos (6 principales + baselines).'],
 'metricas': {'AGB_GEDI': {'r2': 0.8208094196506857,
   'rmse': 13.520098813657226,
   'mae': 10.718364829985916},
  'LAI_MODEL': {'r2': 0.8509745124508388,
   'rmse': 0.15133124210724425,
   'mae': 0.11631065052073573},
  'FMC_MODEL': {'r2': 0.8518247393356425,
   'rmse': 0.34884913869001094,
   'mae': 0.2611771769052744},
  'MLR_baseline': {'r2': 0.19402568009268129,
   'rmse': 28.673643407057323,
   'mae': 22.61080491357414},
  'LightGBM': {'r2': 1.1102230246251565e-16,
   'rmse': 31.939070844504318,
   'mae': 24.320150992030097},
  '3PG_RF': {'r2': 0.09517814462060326,
   'rmse': 30.381122706663394,
   'mae': 23.379725875461293},
  '3PG_LSTM': {'r2': 0.5789020918905111,
   'rmse': 0.6839057246288832,
   'mae': 0.559196320472885},
  'I

## %% [7] Pruebas estadísticas (t-test pareado + bootstrap IC 95%)
Compara estadísticamente el modelo ganador con los demás mediante t-test pareado y calcula intervalos de confianza del 95% por remuestreo bootstrap.

In [30]:
# %% [7] Pruebas estadísticas (t-test pareado + bootstrap IC 95%)
from scipy import stats

registrar_fase("pruebas_estadisticas", "en_curso")


def bootstrap_ic95(y_true, y_pred, n_iter=1000, semilla=CONFIG["SEMILLA"]):
    rng = np.random.default_rng(semilla)
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(y_true)
    r2_boot = []
    for _ in range(n_iter):
        idx = rng.integers(0, n, n)
        try:
            r2_boot.append(r2_score(y_true[idx], y_pred[idx]))
        except Exception:
            continue
    if not r2_boot:
        return {"ic95_bajo": float("nan"), "ic95_alto": float("nan"), "r2_medio_boot": float("nan")}
    return {
        "ic95_bajo": float(np.percentile(r2_boot, 2.5)),
        "ic95_alto": float(np.percentile(r2_boot, 97.5)),
        "r2_medio_boot": float(np.mean(r2_boot)),
    }


pruebas_estadisticas = {}
referencia = "AGB_GEDI"
y_ref = modelos_entrenados[referencia]["y_true"]
pred_ref = modelos_entrenados[referencia]["y_pred"]

for nombre, reg in modelos_entrenados.items():
    if reg.get("objetivo") == "AGBD_gedi":
        err_ref = np.abs(y_ref - pred_ref)
        err_mod = np.abs(reg["y_true"] - reg["y_pred"])
        if nombre == referencia:
            tt = None
        else:
            tt = stats.ttest_rel(err_ref, err_mod)
        pruebas_estadisticas[nombre] = {
            "t_stat": float(tt.statistic) if tt is not None else 0.0,
            "p_valor": float(tt.pvalue) if tt is not None else 1.0,
            "diferencia_significativa": bool(tt.pvalue < 0.05) if tt is not None else False,
            **bootstrap_ic95(reg["y_true"], reg["y_pred"]),
        }
    elif nombre == "INCENDIO":
        pruebas_estadisticas[nombre] = {
            "t_stat": 0.0, "p_valor": 1.0, "diferencia_significativa": False,
            "ic95_bajo": float("nan"), "ic95_alto": float("nan"),
            "r2_medio_boot": float(reg["metricas"].get("roc_auc", float("nan"))),
        }
    else:
        pruebas_estadisticas[nombre] = {
            "t_stat": 0.0, "p_valor": 1.0, "diferencia_significativa": False,
            **bootstrap_ic95(reg["y_true"], reg["y_pred"]),
        }

tests = pruebas_estadisticas  # Alias compatible
registrar_fase("pruebas_estadisticas", "completada", metricas=pruebas_estadisticas)

print("=" * 70)
print("PRUEBAS ESTADÍSTICAS:")
print("=" * 70)
for mod, res in pruebas_estadisticas.items():
    if np.isnan(res["ic95_bajo"]):
        print(f"• {mod:15s}: AUC={res['r2_medio_boot']:.3f}")
    else:
        print(f"• {mod:15s}: IC 95% R²=[{res['ic95_bajo']:.3f}, {res['ic95_alto']:.3f}] | p-val={res['p_valor']:.4f}")


PRUEBAS ESTADÍSTICAS:
• AGB_GEDI       : IC 95% R²=[0.715, 0.859] | p-val=1.0000
• LAI_MODEL      : IC 95% R²=[0.816, 0.881] | p-val=1.0000
• FMC_MODEL      : IC 95% R²=[0.794, 0.886] | p-val=1.0000
• MLR_baseline   : IC 95% R²=[-0.421, 0.432] | p-val=0.0000
• LightGBM       : IC 95% R²=[-0.195, -0.000] | p-val=0.0000
• 3PG_RF         : IC 95% R²=[-0.087, 0.116] | p-val=0.0000
• 3PG_LSTM       : IC 95% R²=[0.437, 0.669] | p-val=1.0000
• INCENDIO       : AUC=0.993


## %% [8] Selección de modelo y tabla comparativa
Genera la tabla comparativa ordenada por R² en validación cruzada y selecciona el mejor modelo, guardando `mejor_modelo.json`.

In [31]:
# %% [8] Selección de modelo y tabla comparativa
registrar_fase("seleccion_modelo", "en_curso")

filas_tabla = []
for nom, reg in modelos_entrenados.items():
    t = pruebas_estadisticas.get(nom, {})
    cv = cv_resultados.get(nom, {})
    es_clf = nom == "INCENDIO"
    filas_tabla.append({
        "modelo": nom,
        "tipo": reg["tipo"],
        "objetivo": reg["objetivo"],
        "r2_train": None if es_clf else round(reg["metricas"]["r2"], 4),
        "rmse_train": None if es_clf else round(reg["metricas"]["rmse"], 2),
        "r2_cv": round(cv.get("r2_medio", float("nan")), 4),
        "rmse_cv": round(cv.get("rmse_medio", float("nan")), 2),
        "extra": (f"AUC={reg['metricas']['roc_auc']:.3f}" if es_clf else ""),
        "ic95_r2": f"[{t.get('ic95_bajo', float('nan')):.3f}, {t.get('ic95_alto', float('nan')):.3f}]",
        "p_valor": round(t.get("p_valor", 1.0), 4),
    })

tabla_comparativa_df = pd.DataFrame(filas_tabla)
tabla = tabla_comparativa_df  # Alias compatible

print("=" * 90)
print("TABLA COMPARATIVA DE MODELOS:")
print("=" * 90)
print(tabla_comparativa_df.to_string(index=False))

# El "mejor modelo" se elige entre los regresores de carbono/biomasa por R² CV
candidatos = tabla_comparativa_df[
    (tabla_comparativa_df["objetivo"].isin(["AGBD_gedi", "NEE"]))
    & (tabla_comparativa_df["r2_cv"].notna())
].copy().sort_values("r2_cv", ascending=False)

mejor_nombre = candidatos.iloc[0]["modelo"]
mejor = modelos_entrenados[mejor_nombre]["objeto"]

metadata_mejor = {
    "mejor_modelo": mejor_nombre,
    "tipo": modelos_entrenados[mejor_nombre]["tipo"],
    "objetivo": modelos_entrenados[mejor_nombre]["objetivo"],
    "metricas": modelos_entrenados[mejor_nombre]["metricas"],
    "validacion_cruzada": cv_resultados.get(mejor_nombre, {}),
    "variables_predictoras": modelos_entrenados[mejor_nombre]["predictoras"],
    "fecha_seleccion": datetime.datetime.utcnow().isoformat() + "Z",
    "fuente_version": CONFIG["FUENTE_VERSION"],
}

ruta_mejor = os.path.join(CONFIG["DIR_MODELOS"], "mejor_modelo.json")
with open(ruta_mejor, "w", encoding="utf-8") as f:
    json.dump(metadata_mejor, f, indent=2, ensure_ascii=False)

registrar_fase("seleccion_modelo", "completada", metricas={"mejor_modelo": mejor_nombre})
print(f"\n🏆 MEJOR MODELO SELECCIONADO: {mejor_nombre} (Guardado en {ruta_mejor})")


TABLA COMPARATIVA DE MODELOS:
      modelo                     tipo  objetivo  r2_train  rmse_train   r2_cv  rmse_cv     extra          ic95_r2  p_valor
    AGB_GEDI                 unico_rf AGBD_gedi    0.8208       13.52 -0.5355    35.14             [0.715, 0.859]      1.0
   LAI_MODEL                 unico_rf       LAI    0.8510        0.15 -0.1061     0.38             [0.816, 0.881]      1.0
   FMC_MODEL                 unico_rf       FMC    0.8518        0.35 -0.6631     0.95             [0.794, 0.886]      1.0
MLR_baseline          baseline_lineal AGBD_gedi    0.1940       28.67 -1.6228    43.77            [-0.421, 0.432]      0.0
    LightGBM        baseline_boosting AGBD_gedi    0.0000       31.94 -0.2533    31.54           [-0.195, -0.000]      0.0
      3PG_RF   hibrido_fisiologico_rf AGBD_gedi    0.0952       30.38 -0.2606    31.77            [-0.087, 0.116]      0.0
    3PG_LSTM hibrido_fisiologico_lstm       NEE    0.5789        0.68  0.5789     0.68             [0.437, 0.

## %% [9] Exportar modelos y artefactos a `modelos_entrenados/`
Exporta los modelos entrenados (`.joblib`, `.h5`), el escalador `StandardScaler`, la lista de columnas predictoras, el historial final y una muestra para Streamlit en `recursos/datos_ejemplo/`.

In [32]:
# %% [9] Exportar modelos y artefactos a modelos_entrenados/
import joblib

registrar_fase("exportacion_artefactos", "en_curso")
log_export = []

# 1. Modelos (LSTM en .h5; resto en .joblib) con su escalador y metadatos
for nombre, reg in modelos_entrenados.items():
    if reg.get("objeto") is None:
        continue
    if reg["tipo"] == "hibrido_fisiologico_lstm":
        ruta_h5 = os.path.join(CONFIG["DIR_MODELOS"], f"{nombre}.h5")
        reg["objeto"].save(ruta_h5)
        joblib.dump(reg["escalador"], os.path.join(CONFIG["DIR_MODELOS"], f"{nombre}_scaler.joblib"))
        log_export.append(f"✓ {nombre} -> {ruta_h5}")
    else:
        ruta = os.path.join(CONFIG["DIR_MODELOS"], f"{nombre}.joblib")
        joblib.dump(reg["objeto"], ruta)
        if reg.get("escalador") is not None:
            joblib.dump(reg["escalador"], os.path.join(CONFIG["DIR_MODELOS"], f"{nombre}_scaler.joblib"))
        log_export.append(f"✓ {nombre} -> {ruta}")
    with open(os.path.join(CONFIG["DIR_MODELOS"], f"{nombre}_metadata.json"), "w", encoding="utf-8") as f:
        json.dump({"tipo": reg["tipo"], "objetivo": reg["objetivo"],
                   "predictoras": reg["predictoras"], "metricas": reg["metricas"]},
                  f, indent=2, ensure_ascii=False, default=str)

# 2. Escalador de predictoras y columnas
joblib.dump(scaler, os.path.join(CONFIG["DIR_MODELOS"], "scaler_predictoras.joblib"))
with open(os.path.join(CONFIG["DIR_MODELOS"], "columnas_predictoras.json"), "w", encoding="utf-8") as f:
    json.dump(PRED_AGB, f, indent=2, ensure_ascii=False)

# 3. Artefactos generales compatibles con entrenamiento_colab (2)
artefactos = {
    "columnas_X": PRED_AGB,
    "fuente_version": CONFIG["FUENTE_VERSION"],
    "mejores_params": mejores_params,
    "escalador": "scaler_predictoras.joblib",
}
joblib.dump(artefactos, os.path.join(CONFIG["DIR_MODELOS"], "artefactos", "artefactos.joblib"))

# 4. Métricas consolidadas
with open(os.path.join(CONFIG["DIR_MODELOS"], "resultados_metricas.json"), "w", encoding="utf-8") as f:
    json.dump({"cv_resultados": cv_resultados, "pruebas_estadisticas": pruebas_estadisticas}, f, indent=2, default=str)

# 5. Muestra para recursos/datos_ejemplo (utilizada por la Pantalla 2 de Streamlit)
muestra = dataset_fusionado.sample(min(20, len(dataset_fusionado)), random_state=CONFIG["SEMILLA"]).copy()
muestra_ruta = os.path.join(CONFIG["DIR_RECURSOS_EJEMPLO"], "muestra_ejemplo.csv")
muestra.to_csv(muestra_ruta, index=False)
log_export.append(f"✓ Muestra Streamlit -> {muestra_ruta}")

registrar_fase("exportacion_artefactos", "completada", mensajes=log_export)
guardar_historial()

for m in log_export:
    print(m)
print(f"\n✓ Todos los artefactos exportados correctamente en: {CONFIG['DIR_MODELOS']}")


✓ AGB_GEDI -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/AGB_GEDI.joblib
✓ LAI_MODEL -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/LAI_MODEL.joblib
✓ FMC_MODEL -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/FMC_MODEL.joblib
✓ MLR_baseline -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/MLR_baseline.joblib
✓ LightGBM -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/LightGBM.joblib
✓ 3PG_RF -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/3PG_RF.joblib
✓ 3PG_LSTM -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/3PG_LSTM.h5
✓ INCENDIO -> /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrenados/INCENDIO.joblib
✓ Muestra Streamlit -> /content/drive/MyDrive/gemelo_digital_forestal/recursos/datos_ejemplo/muestra_ejemplo.csv

✓ Todos los artefactos exportados correctamente en: /content/drive/MyDrive/gemelo_digital_forestal/modelos_entrena

## %% [10] Resumen final y sincronización con el repositorio
Muestra el estado consolidado de todas las fases y genera un archivo ZIP descargable con todos los modelos y artefactos generados.

In [ ]:
# %% [10] Resumen final y compresión ZIP
import shutil

print("=" * 70)
print("RESUMEN GENERAL DEL ENTRENAMIENTO — GEMELO DIGITAL FORESTAL")
print("=" * 70)
print(f"Sitio de estudio       : {CONFIG['SITIO_FLUXNET_ID']} (Tapajós, Amazonía)")
print(f"Periodo de análisis    : {CONFIG['FECHA_INICIO']} a {CONFIG['FECHA_FIN']}")
print(f"Versión de datos       : {CONFIG['FUENTE_VERSION']}")
print(f"Mejor modelo carbono   : {mejor_nombre}")
print("Modelos entrenados:")
for nom, reg in modelos_entrenados.items():
    if nom == "INCENDIO":
        print(f"  • {nom:15s} ({reg['objetivo']}) : AUC={reg['metricas']['roc_auc']:.4f}")
    else:
        print(f"  • {nom:15s} ({reg['objetivo']}) : R²={reg['metricas']['r2']:.4f} | RMSE={reg['metricas']['rmse']:.2f}")
print(f"Fases registradas      : {[f['fase'] for f in HISTORIAL_ENTRENAMIENTO['fases']]}")
print()
print(f"Archivos en {CONFIG['DIR_MODELOS']}:")
for f in sorted(os.listdir(CONFIG["DIR_MODELOS"])):
    print("  •", f)

# Comprimir artefactos generados para descarga o respaldo
carpeta_zip = "gemelo_digital_forestal_artefactos"
shutil.make_archive(carpeta_zip, "zip", ".", CONFIG["DIR_MODELOS"])
print(f"\n✓ Archivo comprimido generado: {carpeta_zip}.zip")

try:
    from google.colab import files
    print("Iniciando descarga automática del archivo ZIP...")
    files.download(f"{carpeta_zip}.zip")
except Exception:
    print(f"ℹ️ Descarga manual: El archivo {carpeta_zip}.zip está disponible en la raíz del entorno de Colab.")
